In [ ]:
# Written by tools/build_notebooks.py -- do not edit. The bootstrap cell below
# compares this against the repo it clones and tells you if these cells are old.
CELLS_SRC = "kaggle_05_interpretability.py"
CELLS_SHA = "da83cfddb99e9d9d"

# 05 - Interpretability: open the box

**Accelerator: GPU T4** (CPU works too). **Runtime: 15-30 minutes. No training.**

This is the project's actual contribution, and it is **forward passes only** on a checkpoint
you already have. Protect this time when the schedule slips.

## The argument, in one sentence

As N grows, the network must partition the **same** 512-filter encoder basis among more
sources - so if mask overlap rises and sparsity falls with N, the degradation curve has a
*mechanistic explanation computed from the network's own internals*, instead of being
asserted.

## The framing that matters

Not "we also built a speaker counter", but **"the count head is an interpretability probe"**.
It produces an explicit, readable estimate of N from the *same* shared features whose mask
geometry we measure. One contribution with two halves beats two half-contributions, for the
same work.

That buys three questions, all answerable here:

1. Does the count head **read** the mask geometry? (correlate confidence against overlap)
2. Do miscounts have a **signature**? (two sources sharing one slot is a mechanistic failure
   explanation, not a shrug)
3. Which basis functions **carry the count**? (ablate filters, watch accuracy fall)

## Before you run

**+ Add Input -> Notebook Output ->** `00_build_dataset` and `02_train`.

## Bootstrap (this cell is identical in every notebook)

Three ways to get the code onto the Kaggle machine, tried in order:

1. **GitHub clone** — set `REPO_URL` below and turn *Internet* ON in the notebook
   settings panel (Settings → Internet → On). This is the recommended route.
2. **Repo-as-dataset** — upload this folder as a Kaggle Dataset called
   `speaker-count-separate` and attach it. No internet needed. Use this if your
   account cannot enable internet (phone-verification is required for that).
3. **Already there** — an existing clone is **fast-forwarded to the newest commit**,
   not reused as-is. A Kaggle session outlives many pushes, and silently running code
   from an hour ago is the most expensive kind of confusion: the log looks fine and the
   fix you are testing is not in it. Any local edits inside the clone are discarded.

Whichever route runs, the commit is printed. Every log can then be traced to the exact
code that produced it.

In [ ]:
REPO_URL = "https://github.com/AlAminAshraf01/speaker-count-separate.git"
REPO_DIR = "/kaggle/working/speaker-count-separate"
REPO_AS_DATASET = "/kaggle/input/speaker-count-separate"

import hashlib
import os
import shutil
import subprocess
import sys


def cells_fingerprint(src_dir: str, name: str) -> str:
    """Short hash of one notebook's percent source plus this shared bootstrap.

    ``tools/build_notebooks.py`` stamps this into every generated ``.ipynb``. The copy
    running on Kaggle recomputes it from the freshly-cloned repo, so a notebook whose
    cells were imported before the last push says so in the first ten seconds instead of
    eleven hours later.

    Line endings are normalised first. The same file is CRLF in a Windows working tree
    and LF in a Linux clone, and a fingerprint that disagrees with itself across
    platforms is worse than no fingerprint at all.
    """
    digest = hashlib.sha256()
    for part in (name, "_bootstrap.py"):
        with open(os.path.join(src_dir, part), "rb") as fh:
            digest.update(fh.read().replace(b"\r\n", b"\n"))
        digest.update(b"\0")
    return digest.hexdigest()[:16]


def cells_status(repo_dir: str, src_name: str | None, stamp: str | None) -> str:
    """Compare the stamp baked into these cells with the repo they are about to run.

    Never raises. A check that can take down every notebook is a worse bug than the one
    it detects, so anything unreadable degrades to "cannot verify".
    """
    if not src_name or not stamp:
        return "unstamped -- re-import this notebook to enable the staleness check"
    try:
        current = cells_fingerprint(os.path.join(repo_dir, "notebooks", "src"), src_name)
    except Exception as exc:
        return f"cannot verify ({exc})"
    if current == stamp:
        return f"current ({stamp})"
    return "\n".join([
        f"STALE  cells {stamp} but repo has {current}",
        "",
        "  These notebook cells were imported before the newest push, so the fix you",
        "  are about to test is not in them. scripts/ and src/ just updated themselves;",
        "  notebook cells cannot, because Kaggle owns them.",
        "",
        "  Fix: File -> Import Notebook -> upload notebooks/" + src_name[:-3] + ".ipynb",
        "       again, re-attach the inputs, and re-run.",
    ])


def _git(repo_dir: str, *argv: str) -> subprocess.CompletedProcess:
    return subprocess.run(["git", "-C", repo_dir, *argv],
                          capture_output=True, text=True)


def update_clone(repo_dir: str) -> str:
    """Fast-forward an existing clone to the remote's newest commit.

    Returns a short status for printing; never raises. Losing internet is a reason to
    carry on with the code that is already there, but it is not a reason to be quiet
    about it -- running stale code unknowingly is how a fix gets tested without being
    present.
    """
    if not os.path.isdir(os.path.join(repo_dir, ".git")):
        return "not a git clone, left as it is"
    branch = _git(repo_dir, "rev-parse", "--abbrev-ref", "HEAD").stdout.strip() or "main"
    before = _git(repo_dir, "rev-parse", "--short", "HEAD").stdout.strip()
    fetched = _git(repo_dir, "fetch", "--depth", "1", "origin", branch)
    if fetched.returncode != 0:
        tail = (fetched.stderr or "").strip().splitlines()
        return f"COULD NOT FETCH ({tail[-1] if tail else 'unknown'}) -- code may be stale"
    reset = _git(repo_dir, "reset", "--hard", f"origin/{branch}")
    if reset.returncode != 0:
        tail = (reset.stderr or "").strip().splitlines()
        return f"COULD NOT UPDATE ({tail[-1] if tail else 'unknown'}) -- code may be stale"
    after = _git(repo_dir, "rev-parse", "--short", "HEAD").stdout.strip()
    return "already newest" if before == after else f"updated {before} -> {after}"


def describe_commit(repo_dir: str) -> str:
    """``<short sha> <date> <subject>`` for the checked-out commit, or a plain note."""
    out = _git(repo_dir, "log", "-1", "--format=%h %cs %s").stdout.strip()
    return out or "no git metadata"


def bootstrap(repo_url: str = REPO_URL, repo_dir: str = REPO_DIR) -> str:
    """Put the repo at `repo_dir`, put its `src/` on sys.path, and chdir into it."""
    if not os.path.isdir(os.path.join(repo_dir, "src")):
        if os.path.isdir(os.path.join(REPO_AS_DATASET, "src")):
            shutil.copytree(REPO_AS_DATASET, repo_dir, dirs_exist_ok=True)
            print(f"copied repo from the attached dataset {REPO_AS_DATASET}")
        else:
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_dir], check=True)
            print(f"cloned {repo_url}")
    else:
        print(f"existing clone: {update_clone(repo_dir)}")
    src = os.path.join(repo_dir, "src")
    if src not in sys.path:
        sys.path.insert(0, src)
    os.chdir(repo_dir)
    return repo_dir


REPO = bootstrap()

import csnet  # noqa: E402

print("csnet", csnet.__version__, "at", REPO)
print("code ", describe_commit(REPO))
# CELLS_SRC / CELLS_SHA are set by the stamp cell that tools/build_notebooks.py puts at
# the top of every generated notebook. globals().get keeps this working in a notebook
# assembled by hand, where that cell may not exist.
CELLS = cells_status(REPO, globals().get("CELLS_SRC"), globals().get("CELLS_SHA"))
print("cells", CELLS if "\n" not in CELLS else "")
if "\n" in CELLS:
    print(CELLS)
print("python", sys.version.split()[0])

import torch  # noqa: E402

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| devices", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  [{i}] {p.name}  {p.total_memory / 1e9:.1f} GB")

In [ ]:
import shlex
import time


def run(cmd: str, check: bool = True) -> int:
    """Run a shell command, streaming its output into the notebook."""
    print("$", cmd, flush=True)
    t0 = time.time()
    proc = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    print(f"\n[exit {code} in {time.time() - t0:.1f}s]", flush=True)
    if check and code != 0:
        raise SystemExit(f"command failed with exit code {code}")
    return code

In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO, "scripts"))
from _common import autodetect_ckpt, autodetect_store
import glob

STORE = autodetect_store()
hits = (glob.glob("/kaggle/input/**/recipes_test.csv", recursive=True)
        + glob.glob(os.path.join(REPO, "data", "recipes_test.csv")))
RECIPES_TEST = hits[0] if hits else None
# By recorded step, so the 30-second dry run cannot win on alphabetical order.
print("checkpoints visible:")
CKPT = autodetect_ckpt()

print("\nstore     :", STORE)
print("checkpoint:", CKPT)
assert CKPT and STORE and RECIPES_TEST, "attach the 00_build_dataset and 02_train outputs"

In [ ]:
run(f"python scripts/12_preflight.py --for interpret"
    f" --store {STORE} --recipes_test {RECIPES_TEST}"
    f" --cells_src {CELLS_SRC} --cells_sha {CELLS_SHA}")

In [ ]:
run(f"python scripts/07_interpret.py"
    f" --ckpt {CKPT}"
    f" --store {STORE}"
    f" --split test"
    f" --recipes {RECIPES_TEST}"
    f" --out /kaggle/working/interpret"
    f" --batch_size 8"
    f" --max_batches 60"
    f" --ablate_batches 20"
    f" --ablate_steps 0 8 16 32 64 128 256")

## Phase 2 deliverable - the learned filterbank

The encoder replaces the STFT with a **learned** 1-D convolutional basis. That is this
project's "custom spectral transformation". Compare the learned centre-frequency curve
against mel and linear spacing: if it is neither, say so and show the figure.

In [ ]:
from IPython.display import Image, display

for name in ["filterbank_time.png", "filterbank_fft.png"]:
    display(Image(f"/kaggle/working/interpret/{name}"))

## Mask geometry as a function of N

Three statistics, all measured from the network's own masks:

* **sparsity** (Hoyer, Gini) - is each source claiming a small part of the basis?
* **pairwise overlap** (cosine, IoU) - are two sources claiming the *same* part?
* **entropy** across slots - how contested is an average time-frequency cell?

In [ ]:
display(Image("/kaggle/working/interpret/mask_stats_vs_n.png"))

In [ ]:
import json
import pandas as pd

report = json.load(open("/kaggle/working/interpret/interpret_report.json"))
df = pd.DataFrame(report["mask_stats_by_n"]).T
df.index.name = "N"
display(df.round(4))
print("\nOverlap and entropy are undefined at N=1 (there is no pair), so NaN there is "
      "correct, not missing data.")

## Does the count head read the geometry?

In [ ]:
display(Image("/kaggle/working/interpret/conf_vs_overlap.png"))

In [ ]:
corr = report["confidence_correlation"]
for name, res in corr.items():
    print(f"corr(count confidence, mask {name:<9s}) r = {res['pearson_r']:+.3f}  "
          f"p = {res['pearson_p']:.3g}  n = {res['n']}")

print("\nmiscount signature:")
display(pd.DataFrame(report["miscount_signature"],
                     columns=["statistic", "count correct", "count wrong"]))

## Which filters carry the count?

Zero the most-active encoder filters and re-measure. A counting accuracy that collapses
faster than separation quality would say the count decision leans on a *specific* part of
the basis - which is the proposal's "extract the filterbank and explain the performance"
deliverable, but with a scalar to attribute against.

In [ ]:
display(Image("/kaggle/working/interpret/filter_ablation.png"))
display(pd.DataFrame(report["ablation"],
                     columns=["filters zeroed", "% of basis", "count acc %",
                              "P-SI-SNR", "SI-SDRi(cc)"]))

## Writing this up

Read your own numbers before choosing the sentence. The honest version depends on what the
figures actually show:

* **If overlap rises and sparsity falls with N** - state that the degradation curve has a
  mechanism: the same basis is being divided among more sources, and the masks show it.
* **If confidence correlates with overlap** - the count head is reading the partition
  geometry, so you can say *why* the model knows how many speakers there are.
* **If miscounted utterances have measurably higher overlap** - two sources sharing one
  output slot is a mechanistic failure explanation.
* **If ablation hurts counting faster than separation** (or the reverse) - report which, and
  note that it localises the count decision in the basis.
* **If a correlation is weak or a trend is flat, say so.** A null result here is still a
  measurement of the network's internals, which is more than an assertion. Do not
  over-claim: with a few hundred utterances, r = 0.1 is noise.

---

This notebook is the part of the project that is *yours* rather than a reproduction. If
quota runs short, cut epochs from `02_train` - not this.